## LangGraph Tutorial

This notebook is organized as a step-by-step tutorial for beginners.
Please run the cells from top to bottom because each section builds on the previous one.

Sections:
1. Setup and API validation
2. Minimal weather graph
3. Stateful weather graph with conditional routing
4. Multi-purpose agent with email and weather handling


In [ ]:
!pip install langgraph langchain openai langchain_openai langchain_community pyowm langchain-google-genai google-generativeai

We will use the Google Weather API to fetch weather information for a location and Gemini as the LLM.

Before running the notebook, set these environment variables in your local environment:
- `GEMINI_API_KEY`
- `GOOGLE_WEATHER_API_KEY`

This notebook intentionally keeps the code tutorial-friendly instead of fully productionized.


In [ ]:
import os

# Read credentials from environment variables instead of hardcoding them in the notebook.
gemini_llm_api = os.getenv("GEMINI_API_KEY", "")
google_weather_api_key = os.getenv("GOOGLE_WEATHER_API_KEY", "")

if not gemini_llm_api or not google_weather_api_key:
    raise ValueError(
        "Please set GEMINI_API_KEY and GOOGLE_WEATHER_API_KEY before running this notebook."
    )

os.environ["GEMINI_API_KEY"] = gemini_llm_api
os.environ["GOOGLE_WEATHER_API_KEY"] = google_weather_api_key


We will use Google Weather API via HTTP requests to get weather information and `ChatGoogleGenerativeAI` for the Gemini model.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import requests

gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.4)


def _geocode_city(city):
    geocode_url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "langgraph-weather-demo/1.0"}
    params = {"q": city, "format": "json", "limit": 1}
    response = requests.get(geocode_url, params=params, headers=headers, timeout=20)
    response.raise_for_status()
    data = response.json()
    if not data:
        raise ValueError(f"Could not resolve location: {city}")
    return float(data[0]["lat"]), float(data[0]["lon"])


def google_weather_lookup(city, units_system="METRIC"):
    api_key = os.environ.get("GOOGLE_WEATHER_API_KEY")
    if not api_key:
        raise ValueError("GOOGLE_WEATHER_API_KEY is not set.")

    lat, lon = _geocode_city(city)
    url = "https://weather.googleapis.com/v1/currentConditions:lookup"
    params = {
        "key": api_key,
        "location.latitude": lat,
        "location.longitude": lon,
        "unitsSystem": units_system,
    }

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return response.json()


### Separate checks for the two services

Run these two cells independently to see whether the Gemini LLM and the weather API are both working.

In [ ]:
try:
    llm_test = gemini_llm.invoke("Say hello in one sentence.")
    print("LLM test passed:", llm_test.content)
except Exception as error:
    print("LLM test failed:", error)


In [ ]:
try:
    weather_test = google_weather_lookup("Taipei")
    condition = weather_test["weatherCondition"]["description"]["text"]
    temp = weather_test["temperature"]["degrees"]
    unit = weather_test["temperature"]["unit"]
    print(f"Weather API test passed: {condition}, {temp} {unit}")
except Exception as error:
    print("Weather API test failed:", error)

In Part 1, we will create the smallest useful LangGraph example:
- one node extracts the city from the user question
- one node fetches weather data
- the graph connects them in sequence


In [ ]:
# Part 1: Minimal weather graph

def extract_city_node_v1(state):
    res = gemini_llm.invoke(f"""
    You are given one question and you have to extract city name from it.
    Do not respond with anything except the city name.
    If you cannot find a city, return an empty string.

    Here is the question:
    {state['input']}
    """)
    return {"city": res.content.strip()}


def weather_lookup_node_v1(state):
    try:
        data = google_weather_lookup(state["city"])
    except Exception as error:
        data = f"Weather service unavailable for {state['city']}: {error}"
    return {"weather": data}


Now let's connect these two nodes and build our first graph.


In [ ]:
from langgraph.graph import StateGraph
from typing import TypedDict


class WeatherWorkflowStateV1(TypedDict):
    input: str
    city: str
    weather: str


workflow = StateGraph(WeatherWorkflowStateV1)
workflow.add_node("extract_city", extract_city_node_v1)
workflow.add_node("weather_lookup", weather_lookup_node_v1)
workflow.add_edge("extract_city", "weather_lookup")


Here is how this first workflow looks. The output of the city extraction node becomes input for the weather node.


In [ ]:
workflow.set_entry_point("extract_city")
workflow.set_finish_point("weather_lookup")

app = workflow.compile()


And finally, let’s run our langgraph!

In [ ]:
app.invoke({"input": "What is weather in Taipei?", "city": "", "weather": ""})

And you will get output like this

```
In Delhi, the current weather is as follows:
Detailed status: haze
Wind speed: 3.6 m/s, direction: 80°
Humidity: 34%
Temperature: 
  - Current: 37.05°C
  - High: 37.05°C
  - Low: 37.05°C
  - Feels like: 39.1°C
Rain: {}
Heat index: None
Cloud cover: 0%
```

But the output is not properly readable, what if we have a responder node that will format this output properly and work as a weather agent instead of a function.

Let’s add a new node called ‘responder’ which will format the weather tool output and provide better results. But wait 🤔, we will need the user’s query to properly format our answer but we will only get the weather tool output in our node so how can we get the user’s query which we are passing to agent node? 👀

This is where states comes into picture. We will create one state called “messages” which will store all the conversation happening in the entire workflow. so let’s create it first!

In [ ]:
# In Part 2, we keep a message history inside state.
from typing import Annotated, Sequence, TypedDict
import operator


class WeatherAgentStateV2(TypedDict):
    messages: Annotated[Sequence[str], operator.add]


Now let's create a responder node so the graph can turn raw weather data into a natural-language answer.


In [ ]:
def responder_node_v2(state):
    response = gemini_llm.invoke(f"""
    You are given weather information and need to answer the user's query clearly.

    User query:
    ---
    {state["messages"][0]}
    ---

    Weather information:
    ---
    {state["messages"][2]}
    ---
    """)
    return {"messages": [response.content]}


Make sure the first two nodes also use state and add their outputs into `messages`.
This shows a more LangGraph-style state flow than passing only one field at a time.


In [ ]:
def extract_city_node_v2(state):
    user_query = state["messages"][0]
    res = gemini_llm.invoke(f"""
    You are given one question and you have to extract city name from it.

    Only reply with the city name if it exists.
    If there is no city name in the question, reply with 'no_response'.

    Here is the question:
    {user_query}
    """)
    return {"messages": [res.content.strip()]}


def weather_lookup_node_v2(state):
    city_name = state["messages"][1]
    try:
        data = google_weather_lookup(city_name)
    except Exception as error:
        data = f"Weather service unavailable for {city_name}: {error}"
    return {"messages": [data]}


Now we can build a stateful version of the graph where every node contributes to the shared message history.


In [ ]:
from langgraph.graph import StateGraph

workflow = StateGraph(WeatherAgentStateV2)
workflow.add_node("extract_city", extract_city_node_v2)
workflow.add_node("weather_lookup", weather_lookup_node_v2)
workflow.add_node("responder", responder_node_v2)

workflow.add_edge("extract_city", "weather_lookup")
workflow.add_edge("weather_lookup", "responder")
workflow.set_entry_point("extract_city")
workflow.set_finish_point("responder")
app = workflow.compile()


This is how the new workflow will look like 👇
![](https://assets-global.website-files.com/62528d398a42420e66390ef9/6641f7d993f204bc6be3def1_image4.png)

Let’s try it with responder!

In [ ]:
inputs = {"messages": ["Hi?"]}
response = app.invoke(inputs)
print(response['messages'][-1])

Now the graph can answer the user more naturally because we added a dedicated responder node.


In [ ]:
from langgraph.graph import END, StateGraph


# Part 3: Add conditional routing so the graph can stop early.
def route_weather_request_v2(state):
    ctx = state["messages"][-1]
    if ctx == "no_response":
        return "end"
    return "continue"


workflow = StateGraph(WeatherAgentStateV2)
workflow.add_node("extract_city", extract_city_node_v2)
workflow.add_node("weather_lookup", weather_lookup_node_v2)
workflow.add_node("responder", responder_node_v2)

workflow.add_conditional_edges("extract_city", route_weather_request_v2, {
    "end": END,
    "continue": "weather_lookup"
})

workflow.add_edge("weather_lookup", "responder")
workflow.set_entry_point("extract_city")
workflow.set_finish_point("responder")
app = workflow.compile()


With this conditional edge, questions like "How are you?" stop early, while weather questions continue to the weather and responder nodes.


## Mini Project: Multi-Purpose AI Agent

Now we will combine the ideas from the earlier sections into a slightly larger tutorial project.
The goal is still educational: show routing, tool usage, and multi-step orchestration in one notebook.


## Workflow
Before creating the project, let’s take a look at the workflow of our agent!
![](https://assets-global.website-files.com/62528d398a42420e66390ef9/6641f8896dd1ef4f1d90e806_image7.png)

We will first add the user input in our entry node where the user input will be categorized into 3 categories:

- email_query: If user want to generate an email response to given email
- weather_query: If user want weather information about any location
- other: If user want any other information

Now based on the categories, we will redirect the query to right node. 🔂

We will use CrewAI to create a crew which can categorize the email and then based on the category it will write a response. We will also create a separate weather agent that uses Google Weather API. For all other queries, we will make a simple Gemini call.

### Prerequisites

Make sure your environment variables are still available before running this section.


In [ ]:
import os

gemini_llm_api = os.getenv("GEMINI_API_KEY", "")
google_weather_api_key = os.getenv("GOOGLE_WEATHER_API_KEY", "")

if not gemini_llm_api or not google_weather_api_key:
    raise ValueError(
        "Please set GEMINI_API_KEY and GOOGLE_WEATHER_API_KEY before running this notebook."
    )

os.environ["GOOGLE_API_KEY"] = gemini_llm_api
os.environ["GEMINI_API_KEY"] = gemini_llm_api
os.environ["GOOGLE_WEATHER_API_KEY"] = google_weather_api_key


With the credentials in place, we can build the multi-purpose agent step by step.


In [ ]:
!pip install langgraph langchain langchain_community pyowm crewai langchain-google-genai google-generativeai

Let's initialize Gemini again for this section and reuse the weather helper from above.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.4)


Let’s import required packages and modules

In [ ]:
%pip install crewai

In [ ]:
# Import required dependencies
from crewai import Crew, Agent, Task
from textwrap import dedent
import os
import json
import requests

Before creating agents or workflows, let’s first define the states which we are going to pass among nodes.

We will use these state variables in our workflow 👇

- messages: It will store the conversation history to keep track of conversation happening in workflow between agents
- email: If user want to generate a email response then entry node will extract the email body from user input and add it in this state
- query: It will store the user’s query
- category: It will store the category of user’s query ( email_query, weather_query or other )

Let’s write the code to define our states:

In [ ]:
from typing import TypedDict


class MultiPurposeAgentState(TypedDict):
    messages: list[str]
    email: str
    query: str
    category: str


First let's define the agents used by the email workflow.


In [ ]:
class EmailAgents:
    def classifier_agent():
        return Agent(
            role="Email Classifier",
            goal="Classify the given email into one of these categories: Important or Casual.",
            backstory="An email classifier that specializes in identifying urgency and tone.",
            verbose=True,
            allow_delegation=False,
        )

    def email_writer_agent():
        return Agent(
            role="Email writing expert",
            goal="Write a reply for Shivam. If the email is important, write professionally. If it is casual, write casually.",
            backstory="An email writer with strong experience in clear and concise replies.",
            verbose=True,
            allow_delegation=False,
        )


Now let's define the tasks for those email agents.


In [ ]:
class EmailTasks:
    def classification_task(agent, email):
        return Task(
            description=dedent(f"""
            You are given an email. Classify this email.
            {email}
            """),
            agent=agent,
            expected_output="Email category as a string",
        )

    def writer_task(agent, email):
        return Task(
            description=dedent(f"""
            Create an email response to the given email based on the category provided by the Email Classifier agent.
            {email}
            """),
            agent=agent,
            expected_output="A concise email response based on the category provided by the Email Classifier agent",
        )


Now we can combine those agents and tasks into a reusable email crew.


In [ ]:
class EmailCrew:
    def __init__(self, email):
        self.email = email

    def run(self):
        classifier_agent = EmailAgents.classifier_agent()
        writer_agent = EmailAgents.email_writer_agent()

        classifier_task = EmailTasks.classification_task(agent=classifier_agent, email=self.email)
        writer_task = EmailTasks.writer_task(agent=writer_agent, email=self.email)

        crew = Crew(
            agents=[classifier_agent, writer_agent],
            tasks=[classifier_task, writer_task],
            verbose=True,
        )
        return crew.kickoff()


Next, create the node that runs the email workflow inside the graph.


In [ ]:
class EmailNodes:
    def writer_node(self, state):
        email = state["email"]
        email_crew = EmailCrew(email)
        crew_result = email_crew.run()
        return {"messages": state["messages"] + [crew_result]}


Now let's define the weather tools and weather agent used by the second branch.


In [ ]:
from langchain.tools import tool


class WeatherTools:
    @tool("Tool to get the weather of any location")
    def weather_tool(location):
        """
        Use this tool when you are given a location and want weather information.
        """
        try:
            data = google_weather_lookup(location)
        except Exception as error:
            data = f"Weather service unavailable for {location}: {error}"
        return data


class MultiPurposeAgents:
    def classifier_agent():
        return Agent(
            role="Email Classifier",
            goal="Classify the given email into one of these categories: Important or Casual.",
            backstory="An email classifier that specializes in identifying urgency and tone.",
            llm="gemini/gemini-2.5-flash",
            verbose=True,
            allow_delegation=False,
        )

    def email_writer_agent():
        return Agent(
            role="Email writing expert",
            goal="Write a reply for Shivam. If the email is important, write professionally. If it is casual, write casually.",
            backstory="An email writer with strong experience in clear and concise replies.",
            llm="gemini/gemini-2.5-flash",
            verbose=True,
            allow_delegation=False,
        )

    def weather_agent():
        return Agent(
            role="Weather Expert",
            goal="Find the weather information for a location using the provided tool.",
            backstory="A weather specialist that uses tools to answer weather-related questions.",
            tools=[WeatherTools.weather_tool],
            llm="gemini/gemini-2.5-flash",
            verbose=True,
            allow_delegation=False,
        )


Let's define the task used by the weather branch.


In [ ]:
class MultiPurposeTasks:
    def classification_task(agent, email):
        return Task(
            description=dedent(f"""
            You are given an email. Classify this email.
            {email}
            """),
            agent=agent,
            expected_output="Email category as a string",
        )

    def writer_task(agent, email):
        return Task(
            description=dedent(f"""
            Create an email response to the given email based on the category provided by the Email Classifier agent.
            {email}
            """),
            agent=agent,
            expected_output="A concise email response based on the category provided by the Email Classifier agent",
        )

    def weather_task(agent, query):
        return Task(
            description=dedent(f"""
            Get the location from the user query and find the weather information about that location.

            Here is the user query:
            {query}
            """),
            agent=agent,
            expected_output="Weather information requested by the user",
        )


Now create the graph node that runs the weather branch.


In [ ]:
class WeatherNodes:
    def weather_node(self, state):
        query = state["query"]
        weather_agent = MultiPurposeAgents.weather_agent()
        weather_task = MultiPurposeTasks.weather_task(agent=weather_agent, query=query)
        result = weather_task.execute()
        return {"messages": state["messages"] + [result]}


Finally, we need an entry node to classify the request and a reply node for anything that does not match email or weather.


In [ ]:
if "gemini_llm" not in globals():
    from langchain_google_genai import ChatGoogleGenerativeAI
    gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.4)


class MultiPurposeNodes:
    def writer_node(self, state):
        email = state["email"]
        email_crew = EmailCrew(email)
        crew_result = email_crew.run()
        return {"messages": state["messages"] + [crew_result]}

    def weather_node(self, state):
        query = state["query"]
        weather_agent = MultiPurposeAgents.weather_agent()
        weather_task = MultiPurposeTasks.weather_task(agent=weather_agent, query=query)
        result = weather_task.execute()
        return {"messages": state["messages"] + [result]}

    def reply_node(self, state):
        query = state["query"]
        response = gemini_llm.invoke(f"""
        {query}
        """)
        return {"messages": state["messages"] + [response.content]}

    def entry_node(self, state):
        user_input = state["query"]
        response = gemini_llm.invoke(f"""
        User input
        ---
        {user_input}
        ---
        You are given one user input and need to choose the next action.

        Categorize the user input into one of these categories:
        email_query: if the user wants to generate a reply to an email
        weather_query: if the user asks for weather information
        other: any other query

        Return valid JSON with these properties only:
        category: category of user input
        email: if category is 'email_query', extract the email body with line breaks, otherwise keep it blank
        query: if category is 'weather_query' or 'other', copy the user query here, otherwise keep it blank
        """)

        raw = (response.content or "").strip()
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            raw = raw[start:end + 1]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            parsed = {"category": "other", "email": "", "query": user_input}

        return {
            "email": parsed.get("email", ""),
            "query": parsed.get("query", user_input),
            "category": parsed.get("category", "other"),
        }


All major nodes are ready. The last step is routing the request to the correct branch.


In [ ]:
def route_multipurpose_request(state):
    category = state["category"]
    print("Category:", category)
    if category == "email_query":
        return "email"
    if category == "weather_query":
        return "weather"
    return "reply"


Now let's assemble the full stateful graph for the mini project.


In [ ]:
from langgraph.graph import END, StateGraph

workflow = StateGraph(MultiPurposeAgentState)
nodes = MultiPurposeNodes()
workflow.add_node("entryNode", nodes.entry_node)
workflow.add_node("weatherNode", nodes.weather_node)
workflow.add_node("responder", nodes.reply_node)
workflow.add_node("emailNode", nodes.writer_node)


Connect the routing function so the graph can decide which branch to execute.


In [ ]:
workflow.add_conditional_edges("entryNode", route_multipurpose_request, {
    "email": "emailNode",
    "weather": "weatherNode",
    "reply": "responder",
})
workflow.add_edge("weatherNode", END)
workflow.add_edge("responder", END)
workflow.add_edge("emailNode", END)

workflow.set_entry_point("entryNode")
app = workflow.compile()


And now it’s time to test our agent 👀!

In [ ]:
query = """
Can you reply to this email

Hello,
Thank you for applying to xyz company
can you share me your previous CTC
Thanks,
HR
"""
inputs = {"query": query, "messages": [query]}
result = app.invoke(inputs)
print("Agent Response:",result['messages'][-1])

After running the above code, you will see the query got categorized as “email_query” and then using EmailCrew it will generate the reply for the extracted email which looks like this:

```
Agent Says: Subject: Re: Application to XYZ Company

Dear [HR's Name],

Thank you for considering my application for the position at XYZ Company.

As per your request, my previous CTC was [mention CTC]. I am open to any negotiation based on the job requirements and the value I can bring to your team at XYZ Company.

Thank you once again for the opportunity. I look forward to potentially furthering the application process.

Best Regards,
[Your Name]
```

Ofc you can make it better with better prompts but I will leave it on you so that you can do experiment with it.

You can also try with below queries for different use cases and agent will reply differently.

weather_query: 
```
Is there any chance of rain in delhi today?
```

email_query:
```
Hey can you generate a reply to this email

Hey man,
I just saw your portfolio and quite liked it. Can you tell me which languages do you use the most
Thanks
```

other:
```
Hello how are you?
```